In [1]:
pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 5.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.2
    Uninstalling pip-24.2:
      Successfully uninstalled pip-24.2
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install torch pandas numpy scikit-learn tqdm kagglehub pennylane matplotlib seaborn --quiet

import os, kagglehub, pandas as pd, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pennylane as qml
from sklearn.metrics import classification_report, confusion_matrix

c:\Users\arpit\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Matplotlib is building the font cache; this may take a moment.


In [ ]:

#  Download dataset

# ================================================
# 🧬 Protein Sequence Dataset - EDA (Top 10 Classes)
# ================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# --- Load Dataset ---
path = kagglehub.dataset_download("anish1137/protein-classification-dataset")
no_dups = os.path.join(path, "pdb_data_no_dups.csv")
seq = os.path.join(path, "pdb_data_seq.csv")

df1 = pd.read_csv(no_dups)
df2 = pd.read_csv(seq)
print("✅ Datasets loaded:", df1.shape, df2.shape)
# ==========================================
# Step 3: Merge & filter top 10 classes
# ==========================================
df = pd.merge(df1[["structureId", "classification"]], df2[["structureId", "sequence"]], on="structureId", how="inner")
print("\n✅ Merged Dataset Shape:", df.shape)

# --- Basic Cleaning ---
df = df.dropna(subset=["sequence", "classification"])
df = df[df["sequence"].apply(lambda x: isinstance(x, str) and len(x.strip()) > 0)]
df["seq_length"] = df["sequence"].apply(len)

print("\n✅ After Cleaning:", df.shape)

# --- Focus on Top 10 Classes ---
top_10 = df["classification"].value_counts().nlargest(10).index
df = df[df["classification"].isin(top_10)].reset_index(drop=True)

print("\n🎯 Filtered to Top 10 Classes:")
print(df["classification"].value_counts())

# --- Sequence Length Distribution ---
plt.figure(figsize=(8,4))
sns.histplot(df["seq_length"], bins=50, kde=True, color="teal")
plt.title("Sequence Length Distribution (Top 10 Classes)")
plt.xlabel("Sequence Length")
plt.ylabel("Frequency")
plt.show()

# --- Top 10 Protein Classes Count ---
plt.figure(figsize=(10,4))
sns.barplot(x=df["classification"].value_counts().index,
            y=df["classification"].value_counts().values,
            palette="crest")
plt.title("Protein Class Distribution (Top 10)")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Count")
plt.show()

# --- Amino Acid Composition ---
AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")
aa_counts = Counter("".join(df["sequence"]))
aa_freq = {aa: aa_counts.get(aa, 0) for aa in AA_LIST}

plt.figure(figsize=(10,4))
sns.barplot(x=list(aa_freq.keys()), y=list(aa_freq.values()), palette="mako")
plt.title("Amino Acid Frequency (Top 10 Classes Dataset)")
plt.xlabel("Amino Acid")
plt.ylabel("Count")
plt.show()

# --- Sequence Length per Class ---
plt.figure(figsize=(10,4))
sns.boxplot(data=df, x="classification", y="seq_length", palette="viridis")
plt.title("Sequence Length per Protein Class (Top 10)")
plt.xticks(rotation=45, ha="right")
plt.show()

# --- Save Cleaned Dataset ---
df.to_csv("protein_top10_cleaned.csv", index=False)
print("\n✅ Cleaned Top 10 Class dataset saved as 'protein_top10_cleaned.csv'")
print("📊 Final shape:", df.shape)


In [4]:
import torch

# ==========================================
# Step 1: Device setup
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

✅ Using device: cpu


In [5]:
# ==========================================
# Step 4: Encode sequences
# ==========================================
def encode_sequence(seq, max_len=300):
    seq = str(seq)
    encoded = [ord(c) % 20 for c in seq[:max_len]]
    encoded += [0] * (max_len - len(encoded))
    return encoded

df["encoded_seq"] = df["sequence"].apply(encode_sequence)

le = LabelEncoder()
df["encoded_label"] = le.fit_transform(df["classification"])

X = np.stack(df["encoded_seq"].values)
y = df["encoded_label"].values

In [6]:
# ==========================================
# Step 5: Train-test split
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = torch.tensor(X_train, dtype=torch.float64).unsqueeze(1)
X_test = torch.tensor(X_test, dtype=torch.float64).unsqueeze(1)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

from torch.utils.data import DataLoader, TensorDataset
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64)

In [7]:
import math
import numpy as np
import pandas as pd
from typing import List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import pennylane as qml

# -------------------------
# AA mapping and preprocessing
# -------------------------
AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")  # 20 canonical amino acids
AA_TO_IDX = {aa: i+1 for i, aa in enumerate(AA_LIST)}  # 0 reserved for pad/unknown

def seq_to_indices(seq: str, max_len: int):
    seq = str(seq).upper().strip()
    indices = [AA_TO_IDX.get(ch, 0) for ch in seq[:max_len]]
    if len(indices) < max_len:
        indices += [0] * (max_len - len(indices))
    return indices

class ProteinSequenceDataset(Dataset):
    def __init__(self, sequences: List[str], labels: List[int], max_len: int = 300):
        assert len(sequences) == len(labels)
        self.max_len = max_len
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        indices = np.array(seq_to_indices(seq, self.max_len), dtype=np.int64)
        return torch.from_numpy(indices), torch.tensor(label, dtype=torch.long)

# -------------------------
# Classical encoders
# -------------------------
class CNNEncoder(nn.Module):
    def __init__(self, num_embeddings=21, emb_dim=64, out_dim=128, kernel_sizes=(3,5,7,11)):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=num_embeddings, embedding_dim=emb_dim, padding_idx=0)

        num_convs = len(kernel_sizes)
        # Calculate output channels per convolution. This ensures an integer value.
        out_channels_per_conv = out_dim // num_convs

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=emb_dim, out_channels=out_channels_per_conv, kernel_size=k, padding=k//2)
            for k in kernel_sizes
        ])
        # The actual output dimension will be the sum of out_channels_per_conv from all convs.
        self.output_dim = out_channels_per_conv * num_convs
        self.pool = nn.AdaptiveMaxPool1d(1)

    def forward(self, x):
        emb = self.embedding(x)            # (batch, seq_len, emb_dim)
        emb = emb.permute(0,2,1)          # (batch, emb_dim, seq_len)
        conv_outs = []
        for conv in self.convs:
            c = conv(emb)                 # (batch, out_ch, seq_len)
            c = F.relu(c)
            c = self.pool(c).squeeze(-1)  # (batch, out_ch)
            conv_outs.append(c)
        out = torch.cat(conv_outs, dim=1) # (batch, output_dim)
        return out


# -------------------------
# Quantum layer (PennyLane)
# -------------------------
def create_quantum_torchlayer(n_qubits=10, n_layers=5, dev_name='default.qubit', shots=None):
    dev = qml.device(dev_name, wires=n_qubits, shots=shots)

    @qml.qnode(dev, interface='torch', diff_method='backprop')
    def qnode(inputs, weights):
        for i in range(n_qubits):
            # Inputs are expected to be batched if diff_method='backprop'
            qml.RX(inputs[:, i], wires=i)
        k = 0
        for l in range(n_layers):
            for i in range(n_qubits):
                qml.Rot(weights[k,0], weights[k,1], weights[k,2], wires=i)
                k += 1
            for i in range(n_qubits - 1):
                qml.CNOT(wires=[i, i+1])
            qml.CNOT(wires=[n_qubits-1, 0])
        expvals = [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        # For diff_method='backprop', qml.expval returns batched values.
        # TorchLayer expects a list of these batched expectation values.
        return expvals

    weight_shapes = {"weights": (n_layers * n_qubits, 3)}
    qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)
    return qlayer

# -------------------------
# Hybrid Model
# -------------------------
# --- FIXED HYBRID VQNN MODEL (auto-detects encoder output dim) ---
class HybridVQNN(nn.Module):
    def __init__(self,
                 encoder: nn.Module,
                 n_qubits: int = 10,
                 q_layers: int = 4,
                 classical_to_quantum_dim: int = None,
                 num_classes: int = 10,
                 use_compressor=True):
        super().__init__()
        self.encoder = encoder

        # Auto-detect encoder output dimension
        with torch.no_grad():
            dummy = torch.zeros(1, 512, dtype=torch.long)
            enc_dim = self.encoder(dummy).shape[1]

        if classical_to_quantum_dim is None:
            classical_to_quantum_dim = n_qubits
        self.use_compressor = use_compressor
        if use_compressor:
            self.compressor = nn.Sequential(
                nn.Linear(enc_dim, max(enc_dim // 2, classical_to_quantum_dim)),
                nn.ReLU(),
                nn.Linear(max(enc_dim // 2, classical_to_quantum_dim), classical_to_quantum_dim),
                nn.Tanh()
            )
        else:
            if enc_dim != classical_to_quantum_dim:
                raise ValueError("If not using compressor, enc_dim must equal classical_to_quantum_dim")

        self.qlayer = create_quantum_torchlayer(n_qubits=n_qubits, n_layers=q_layers)


        self.classical_head = nn.Sequential(
            nn.Linear(n_qubits, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        enc = self.encoder(x)
        if self.use_compressor:
            compressed = self.compressor(enc)
            q_in = compressed * math.pi
        else:
            q_in = enc
        q_out = self.qlayer(q_in)
        logits = self.classical_head(q_out)
        return logits
print("✅ Fixed HybridVQNN class loaded successfully.")

# -------------------------
# Training / Eval helpers
# -------------------------
def train_epoch(model, device, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for x_batch, y_batch in tqdm(loader, desc="Train batches", leave=False):
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x_batch.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total += x_batch.size(0)
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def eval_epoch(model, device, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x_batch, y_batch in tqdm(loader, desc="Eval batches", leave=False):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item() * x_batch.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += x_batch.size(0)
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# -------------------------
# Utility: prepare dataloaders from CSV or dataframe
# -------------------------
def prepare_loaders_from_df(df, seq_col='sequence', label_col='classification', max_len=300, batch_size=32, val_split=0.2):
    # make sure labels are integer encoded 0..C-1
    if df[label_col].dtype == object or not pd.api.types.is_integer_dtype(df[label_col].dtype):
        df[label_col] = pd.Categorical(df[label_col]).codes
    sequences = df[seq_col].fillna("").tolist()
    labels = df[label_col].astype(int).tolist()
    num_samples = len(sequences)
    idxs = np.arange(num_samples)
    np.random.seed(42)
    np.random.shuffle(idxs)
    split = int((1 - val_split) * num_samples)
    train_idx = idxs[:split]
    val_idx = idxs[split:]
    train_seqs = [sequences[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]
    val_seqs = [sequences[i] for i in val_idx]
    val_labels = [labels[i] for i in val_idx]
    train_ds = ProteinSequenceDataset(train_seqs, train_labels, max_len=max_len)
    val_ds = ProteinSequenceDataset(val_seqs, val_labels, max_len=max_len)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, len(np.unique(labels))



✅ Fixed HybridVQNN class loaded successfully.


In [8]:
# --- TRAINING & EVALUATION SECTION ---
# This cell assumes you already have a dataframe named `df`
# with columns: 'sequence' and 'classification'

# Prepare data loaders
train_loader, val_loader, num_classes = prepare_loaders_from_df(
    df, seq_col='sequence', label_col='classification',
    max_len=300, batch_size=128, val_split=0.2
)

# Initialize model, loss, optimizer
encoder = CNNEncoder(num_embeddings=21, emb_dim=64, out_dim=128)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HybridVQNN(
    encoder=encoder,
    n_qubits=10,
    q_layers=5,
    classical_to_quantum_dim=10,
    num_classes=num_classes,
    use_compressor=True
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# Train model
num_epochs = 30
for epoch in range(1, num_epochs + 1):
    print(f"\nEpoch {epoch}/{num_epochs}")
    train_loss, train_acc = train_epoch(model, device, train_loader, criterion, optimizer)
    val_loss, val_acc = eval_epoch(model, device, val_loader, criterion)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

# Save model if needed
torch.save(model.state_dict(), "hybrid_qnn_protein_model.pt")
print("✅ Training completed and model saved as 'hybrid_qnn_protein_model.pt'")


Epoch 1/30


Train Loss: 1.6011 | Train Acc: 0.3885
Val   Loss: 1.2360 | Val   Acc: 0.5154

Epoch 2/30


Train Loss: 1.1083 | Train Acc: 0.5901
Val   Loss: 0.9552 | Val   Acc: 0.6741

Epoch 3/30


Train Loss: 0.8866 | Train Acc: 0.6997
Val   Loss: 0.8761 | Val   Acc: 0.7053

Epoch 4/30


Train Loss: 0.7773 | Train Acc: 0.7426
Val   Loss: 0.7742 | Val   Acc: 0.7522

Epoch 5/30


...

Train Loss: 0.4133 | Train Acc: 0.8740
Val   Loss: 0.4709 | Val   Acc: 0.8645

Epoch 27/30


Train Loss: 0.4085 | Train Acc: 0.8754
Val   Loss: 0.4798 | Val   Acc: 0.8638

Epoch 28/30


Train Loss: 0.4073 | Train Acc: 0.8765
Val   Loss: 0.4671 | Val   Acc: 0.8647

Epoch 29/30


Train Loss: 0.4033 | Train Acc: 0.8764
Val   Loss: 0.4781 | Val   Acc: 0.8625

Epoch 30/30


Train Loss: 0.3976 | Train Acc: 0.8784
Val   Loss: 0.4675 | Val   Acc: 0.8655
✅ Training completed and model saved as 'hybrid_qnn_protein_model.pt'


In [8]:
encoder = CNNEncoder(num_embeddings=21, emb_dim=64, out_dim=128)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = 10 # There are 10 top classes as determined during data loading
model = HybridVQNN(
    encoder=encoder,
    n_qubits=10,
    q_layers=4,
    classical_to_quantum_dim=10,
    num_classes=num_classes,
    use_compressor=True
).to(device)

model.load_state_dict(torch.load("hybrid_qnn_protein_model.pt", map_location=device))
model.eval()

print("✅ Model loaded successfully!")


✅ Model loaded successfully!


In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder # Ensure LabelEncoder is imported

sequence = "PPYTVVYFPVRGRCAALRMLLADQGQSWKEEVVTVETWQEGSLKASCLYGQLPKFQDGDLTLYQSNTILRHLGRTLGLYGKDQQEAALVDMVNDGVEDLRCKYISLIYTNYEAGKDDYVKALPGQLKPFETLLSQNQGGKTFIVGDQISFADYNLLDLLLIHEVLAPGCLDAFPLLSAYVGRLSARPKLKAFLASPEYVNLPINGNGKQ" # <-- Modified sequence example
x = torch.tensor(seq_to_indices(sequence, max_len=300), dtype=torch.long).unsqueeze(0).to(device) # unsqueeze to add batch dimension

with torch.no_grad():
    output = model(x)
    predicted_class_idx = output.argmax(dim=1).item()

le_final = LabelEncoder()
le_final.fit(df["classification"])
predicted_class_name = le_final.inverse_transform([predicted_class_idx])[0]
print(f"Predicted class for the sequence: {predicted_class_name} (Index: {predicted_class_idx})")



Predicted class for the sequence: TRANSFERASE (Index: 7)


In [11]:
print("Classification Labels and their Indices:")
for i, class_name in enumerate(le.classes_):
    print(f"  {class_name}: {i}")

Classification Labels and their Indices:
  HYDROLASE: 0
  HYDROLASE/HYDROLASE INHIBITOR: 1
  IMMUNE SYSTEM: 2
  LYASE: 3
  OXIDOREDUCTASE: 4
  RIBOSOME: 5
  TRANSCRIPTION: 6
  TRANSFERASE: 7
  VIRAL PROTEIN: 8
  VIRUS: 9
